# 02 - Data Preprocessing

## Objective

The purpose of this notebook is to prepare the California CRMLS Sold dataset for machine learning.

The preprocessing workflow includes:

- Loading the filtered residential dataset.
- Handling missing values.
- Encoding categorical variables.
- Normalizing numerical features when appropriate.
- Creating a time-based training and testing split.
- Exporting the cleaned dataset for model development.

The resulting dataset will be used as the input for the baseline predictive models developed in the next stage of the project.

In [129]:
# ============================================
# Import required libraries
# ============================================

from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

# Display all columns when inspecting DataFrames
pd.set_option("display.max_columns", None)

## Load the Combined Dataset

The exploratory analysis produced a filtered dataset containing residential single-family property sales.

In this section, we will load that dataset and verify that it has been imported successfully before beginning the preprocessing steps.

In [130]:
# ============================================
# Locate all monthly CSV files
# ============================================

data_path = Path("../data/California")

csv_files = sorted(data_path.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files.")

# ============================================
# Read each monthly CSV file
# ============================================

dataframes = []

for file in csv_files:
    print(f"Loading {file.name}...")
    df = pd.read_csv(file, low_memory=False)
    dataframes.append(df)

# Combine all monthly datasets into one DataFrame
homes = pd.concat(dataframes, ignore_index=True)

print("\nAll files successfully combined!")

# ============================================
# Filter to Residential Single-Family properties
# ============================================

homes = homes[
    (homes["PropertyType"] == "Residential") &
    (homes["PropertySubType"] == "SingleFamilyResidence")
].copy()

# Display basic information about the filtered dataset
print(f"\nRows:    {homes.shape[0]:,}")
print(f"Columns: {homes.shape[1]}")

homes.head()

Found 30 CSV files.
Loading CRMLSSold20220101_20231231_filled.csv...
Loading CRMLSSold202401_filled.csv...
Loading CRMLSSold202402_filled.csv...
Loading CRMLSSold202403_filled.csv...
Loading CRMLSSold202404_filled.csv...
Loading CRMLSSold202405_filled.csv...
Loading CRMLSSold202406_filled.csv...
Loading CRMLSSold202407_filled.csv...
Loading CRMLSSold202408.csv...
Loading CRMLSSold202409.csv...
Loading CRMLSSold202410.csv...
Loading CRMLSSold202411.csv...
Loading CRMLSSold202412.csv...
Loading CRMLSSold202501_filled.csv...
Loading CRMLSSold202502.csv...
Loading CRMLSSold202503.csv...
Loading CRMLSSold202504.csv...
Loading CRMLSSold202505.csv...
Loading CRMLSSold202506.csv...
Loading CRMLSSold202507.csv...
Loading CRMLSSold202508.csv...
Loading CRMLSSold202509.csv...
Loading CRMLSSold202510.csv...
Loading CRMLSSold202511.csv...
Loading CRMLSSold202512.csv...
Loading CRMLSSold202601.csv...
Loading CRMLSSold202602.csv...
Loading CRMLSSold202603.csv...
Loading CRMLSSold202604.csv...
Loading

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,BuyerOfficeName,CoListOfficeName,ListAgentFullName,CoListAgentFirstName,CoListAgentLastName,BuyerAgentMlsId,BuyerAgentFirstName,BuyerAgentLastName,FireplacesTotal,AssociationFeeFrequency,AboveGradeFinishedArea,ListingKeyNumeric,MLSAreaMajor,TaxAnnualAmount,CountyOrParish,MlsStatus,ElementarySchool,AttachedGarageYN,ParkingTotal,BuilderName,PropertySubType,LotSizeAcres,SubdivisionName,BuyerOfficeAOR,YearBuilt,BuyerAgencyCompensationType,StreetNumberNumeric,ListingId,BathroomsTotalInteger,City,BuyerAgencyCompensation,TaxYear,BuildingAreaTotal,BedroomsTotal,ContractStatusChangeDate,ElementarySchoolDistrict,CoBuyerAgentFirstName,PurchaseContractDate,ListingContractDate,BelowGradeFinishedArea,BusinessType,StateOrProvince,CoveredSpaces,MiddleOrJuniorSchool,FireplaceYN,Stories,HighSchool,Levels,LotSizeDimensions,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,BuyerAgentAOR,ListAgentAOR
3,NaN,True,NaN,NaN,False,2499999.0,556365765,bryanmeathe@gmail.com,2022-01-04,2499999.0,Bryan,Meathe,33.147270,-117.340604,295 Chinquapin Ave,Residential,2645.0,2499999.0,37.0,Coldwell Banker Realty,ERA Ranch & Sea Realty,NaN,Bryan Meathe,NaN,NaN,101046,Bill,Kellaway,NaN,Monthly,NaN,556365765.0,92008 - Carlsbad,NaN,San Diego,Closed,NaN,True,2.0,NaN,SingleFamilyResidence,0.3071,NaN,NorthSanDiegoCounty,2016.0,Item1,295.0,NDP2111696,4.0,Carlsbad,2.25,NaN,NaN,4.0,2022-01-04,NaN,NaN,2021-11-19,2021-10-13,NaN,NaN,CA,NaN,NaN,False,NaN,NaN,ThreeOrMore,NaN,13376.0,0.0,NaN,2.0,Carlsbad Unified,92008,140.0,13376.0,NaN,False,False,NaN,NaN
6,NaN,True,NaN,NaN,False,565000.0,556365286,joseph@labelrealtygroup.com,2022-01-10,640000.0,Joseph,Guzman,34.235979,-117.202269,944 Mecham Drive,Residential,2070.0,565000.0,13.0,Label Realty Group,IRN Realty,NaN,Joseph Guzman,NaN,NaN,HAGUIREN,Rene,Aguilar,NaN,NaN,NaN,556365286.0,287 - Arrowhead Area,NaN,San Bernardino,Closed,NaN,True,1.0,NaN,SingleFamilyResidence,0.0780,NaN,TriCounties,2007.0,Item1,944.0,OC21227640,3.0,Lake Arrowhead,2.00,NaN,NaN,3.0,2022-01-10,NaN,NaN,2021-10-26,2021-10-12,NaN,NaN,CA,NaN,NaN,True,NaN,NaN,ThreeOrMore,NaN,3397.0,0.0,False,1.0,Rim of the World,92352,0.0,3397.0,NaN,False,False,NaN,NaN
7,NaN,False,NaN,NaN,False,349999.0,556365284,eaguayo08@gmail.com,2022-03-23,438000.0,ELIAS,AGUAYO,34.114130,-117.442493,16596 Miller Avenue,Residential,1174.0,349999.0,3.0,INTERNATIONAL REAL ESTATE SVCS,INTERNATIONAL REAL ESTATE SVCS,NaN,ELIAS AGUAYO,NaN,NaN,C32048,ELIAS,AGUAYO,NaN,NaN,NaN,556365284.0,264 - Fontana,NaN,San Bernardino,Closed,NaN,True,2.0,NaN,SingleFamilyResidence,0.2273,NaN,CitrusValley,1960.0,Item1,16596.0,CV21227637,2.0,Fontana,2.00,NaN,NaN,3.0,2022-03-23,NaN,NaN,2021-10-16,2021-10-13,NaN,NaN,CA,NaN,NaN,False,1.0,NaN,One,NaN,9900.0,3.0,False,2.0,Fontana Unified,92336,0.0,9900.0,NaN,False,False,NaN,NaN
12,"Carpet,Tile",True,NaN,NaN,True,599000.0,556364261,gcrb714@gmail.com,2022-01-10,615000.0,Rhonda,Belous,33.783767,-116.447280,36353 Paseo Del Sol,Residential,1996.0,599000.0,81.0,"Whitestar Management, Inc",RE/MAX Consultants,"Whitestar Management, Inc",Rhonda Belous,Patrick,Belous,CDAR-43944,Dawn,Jonker,NaN,Monthly,NaN,556364261.0,336 - Cathedral City South,NaN,Riverside,Closed,NaN,False,4.0,NaN,SingleFamilyResidence,0.1400,Rio Del Sol,CaliforniaDesert,2005.0,Item1,36353.0,219068900DA,2.0,Cathedral City,2.00,NaN,NaN,2.0,2022-01-10,NaN,NaN,2022-01-04,2021-10-13,NaN,NaN,CA,NaN,NaN,True,NaN,NaN,NaN,NaN,6098.0,NaN,NaN,2.0,NaN,92234,195.0,6098.0,NaN,False,False,NaN,NaN
14,NaN,True,NaN,NaN,False,399990.0,556363890,allegra@3treehomes.com,2022-01-19,399990.0,Allegra,Gigante-Luft,39.767729,-121.586400,6091 Vista Knolls Drive,Residential,14

## Handle Missing Values

Before preparing the data for machine learning, we will examine the amount of missing data across the dataset.

Missing values can negatively affect model performance, so it is important to determine an appropriate strategy for each variable. Depending on the amount and importance of the missing data, we may:

- Drop observations with missing values.
- Impute missing values using an appropriate statistic.
- Leave variables unchanged if missing values are negligible.

The decisions made in this section will be documented and used consistently throughout the remainder of the project.

In [131]:
# ============================================
# Check missing values across the entire dataset
# ============================================

missing = homes.isnull().sum().to_frame(name="Missing Values")

missing["Percent Missing"] = (
    missing["Missing Values"] / len(homes) * 100
).round(2)

# Display only columns with missing values
missing = missing[missing["Missing Values"] > 0]

# Sort from most missing to least missing
missing = missing.sort_values(
    by="Percent Missing",
    ascending=False
)

missing

,Missing Values,Percent Missing
BusinessType,399157,100.00
MiddleOrJuniorSchoolDistrict,399157,100.00
FireplacesTotal,399157,100.00
TaxAnnualAmount,399157,100.00
AboveGradeFinishedArea,399157,100.00
...,...,...
BathroomsTotalInteger,75,0.02
ListAgentLastName,37,0.01
ClosePrice,2,0.00
StateOrProvince,1,0.00


## Remove Empty Columns

Some variables contain missing values for every observation.

Because these columns contain no usable information, they will be removed before
continuing the preprocessing workflow. The remaining variables will then be
evaluated individually to determine the appropriate strategy for handling any
remaining missing values.

In [132]:
# ============================================
# Remove columns with 100% missing values
# ============================================

# Identify columns where every value is missing
empty_columns = homes.columns[homes.isnull().all()]

print(f"Columns with 100% missing values: {len(empty_columns)}")

# Display their names
for col in empty_columns:
    print(col)

# Remove them
homes = homes.drop(columns=empty_columns)

print(f"\nRemaining columns: {homes.shape[1]}")

Columns with 100% missing values: 8
FireplacesTotal
AboveGradeFinishedArea
TaxAnnualAmount
TaxYear
ElementarySchoolDistrict
BusinessType
CoveredSpaces
MiddleOrJuniorSchoolDistrict

Remaining columns: 74


## Handle Remaining Missing Values

After removing columns with no usable information, we will examine the remaining
variables that still contain missing values.

Rather than applying a single strategy to every variable, we will identify which
columns contain missing data and determine the most appropriate treatment based
on the amount of missing information and the role of each variable in the model.

In [133]:
# ============================================
# Summarize remaining missing values
# ============================================

missing = homes.isnull().sum().to_frame(name="Missing Values")

missing["Percent Missing"] = (
    missing["Missing Values"] / len(homes) * 100
).round(2)

# Keep only columns that still contain missing values
missing = missing[missing["Missing Values"] > 0]

# Sort from highest to lowest percentage missing
missing = missing.sort_values(
    by="Percent Missing",
    ascending=False
)

missing

,Missing Values,Percent Missing
WaterfrontYN,398982,99.96
BelowGradeFinishedArea,396829,99.42
BasementYN,389410,97.56
BuilderName,380399,95.30
LotSizeDimensions,374649,93.86
...,...,...
BathroomsTotalInteger,75,0.02
ListAgentLastName,37,0.01
ClosePrice,2,0.00
StateOrProvince,1,0.00


## Remove Variables with Excessive Missing Data

Some variables still contain missing values for the majority of observations.

For the baseline model, variables with more than 50% missing values will be removed.
Retaining these variables would require extensive imputation while contributing
relatively little information to the predictive model.

The remaining variables contain sufficient data to support preprocessing and
model development.

In [134]:
# ============================================
# Remove columns with more than 50% missing values
# ============================================

# Remove variables with more than 50% missing values
# to simplify the baseline model.
missing_threshold = 50

columns_to_drop = missing[
    missing["Percent Missing"] > missing_threshold
].index

print(f"Columns removed: {len(columns_to_drop)}")

for column in columns_to_drop:
    print(column)

homes = homes.drop(columns=columns_to_drop)

print(f"\nRemaining columns: {homes.shape[1]}")

Columns removed: 19
WaterfrontYN
BelowGradeFinishedArea
BasementYN
BuilderName
LotSizeDimensions
BuildingAreaTotal
CoBuyerAgentFirstName
ElementarySchool
MiddleOrJuniorSchool
HighSchool
CoListAgentFirstName
CoListAgentLastName
CoListOfficeName
AssociationFeeFrequency
BuyerAgencyCompensation
BuyerAgencyCompensationType
SubdivisionName
latfilled
lonfilled

Remaining columns: 55


## Impute Remaining Missing Values

After removing variables with excessive missing data, the remaining dataset still
contains a small number of missing values.

Different data types require different imputation strategies:

- Numerical variables will be imputed using the median value.
- Categorical variables will be imputed using the most frequent value (mode).

These approaches preserve all observations while minimizing the impact of missing
data on the baseline machine learning models.

In [135]:
# ============================================
# Impute remaining missing values
# ============================================

# Identify numerical and categorical columns
numeric_columns = homes.select_dtypes(include=["number"]).columns
categorical_columns = homes.select_dtypes(
    include=["object", "string"]
).columns

# Impute numerical columns with the median
homes[numeric_columns] = homes[numeric_columns].fillna(
    homes[numeric_columns].median()
)

# Impute categorical columns with the mode
for column in categorical_columns:
    homes[column] = homes[column].fillna(
        homes[column].mode()[0]
    )

# Verify that missing values have been addressed
remaining_missing = homes.isnull().sum().sum()

print(f"Remaining missing values: {remaining_missing:,}")

Remaining missing values: 0


## Encode Categorical Variables

Machine learning models require numerical input and cannot directly interpret
text-based categories.

In this section, categorical variables will be converted into numerical
representations so they can be used during model training.

For the baseline model, one-hot encoding will be used to represent categorical
variables while avoiding any artificial ordering between categories.

In [136]:
# ============================================
# Examine categorical variables
# ============================================

categorical_columns = homes.select_dtypes(
    include=["object", "string"]
).columns

for column in categorical_columns:
    print(f"{column}: {homes[column].nunique():,} unique values")

Flooring: 340 unique values
ViewYN: 2 unique values
PoolPrivateYN: 2 unique values
ListAgentEmail: 83,262 unique values
CloseDate: 1,424 unique values
ListAgentFirstName: 15,154 unique values
ListAgentLastName: 34,303 unique values
UnparsedAddress: 384,499 unique values
PropertyType: 1 unique values
ListOfficeName: 18,941 unique values
BuyerOfficeName: 21,819 unique values
ListAgentFullName: 77,106 unique values
BuyerAgentMlsId: 115,421 unique values
BuyerAgentFirstName: 19,225 unique values
BuyerAgentLastName: 39,937 unique values
MLSAreaMajor: 1,093 unique values
CountyOrParish: 64 unique values
MlsStatus: 1 unique values
AttachedGarageYN: 2 unique values
PropertySubType: 1 unique values
BuyerOfficeAOR: 74 unique values
ListingId: 398,881 unique values
City: 1,155 unique values
ContractStatusChangeDate: 1,421 unique values
PurchaseContractDate: 2,025 unique values
ListingContractDate: 2,300 unique values
StateOrProvince: 14 unique values
FireplaceYN: 2 unique values
Levels: 18 unique

## Remove Identifier and Administrative Variables

Some variables uniquely identify a property, listing, office, or agent rather than
describing the property's characteristics.

These variables are removed because they are not expected to improve prediction of
home prices and would unnecessarily increase the complexity of the dataset.

In [137]:
# ============================================
# Remove identifier and administrative columns
# ============================================

columns_to_drop = [
    "ListingId",
    "UnparsedAddress",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFullName",
    "ListAgentEmail",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "BuyerAgentMlsId",
    "ListOfficeName",
    "BuyerOfficeName",
]

homes = homes.drop(columns=columns_to_drop)

print(f"Remaining columns: {homes.shape[1]}")

Remaining columns: 44


In [138]:
# ============================================
# Display remaining categorical columns
# ============================================

categorical_columns = homes.select_dtypes(
    include=["object", "string"]
).columns

for column in categorical_columns:
    print(f"{column}: {homes[column].nunique()} unique values")

Flooring: 340 unique values
ViewYN: 2 unique values
PoolPrivateYN: 2 unique values
CloseDate: 1424 unique values
PropertyType: 1 unique values
MLSAreaMajor: 1093 unique values
CountyOrParish: 64 unique values
MlsStatus: 1 unique values
AttachedGarageYN: 2 unique values
PropertySubType: 1 unique values
BuyerOfficeAOR: 74 unique values
City: 1155 unique values
ContractStatusChangeDate: 1421 unique values
PurchaseContractDate: 2025 unique values
ListingContractDate: 2300 unique values
StateOrProvince: 14 unique values
FireplaceYN: 2 unique values
Levels: 18 unique values
NewConstructionYN: 2 unique values
HighSchoolDistrict: 446 unique values
PostalCode: 3678 unique values
BuyerAgentAOR: 62 unique values
ListAgentAOR: 61 unique values


## Remove Constant and Unused Date Variables

Some variables contain only a single unique value after filtering, while others
represent dates that are not required for the baseline predictive model.

These variables are removed to simplify the dataset before encoding the remaining
categorical features.

In [139]:
# ============================================
# Remove constant and unused date columns
# ============================================

columns_to_drop = [
    "PropertyType",
    "PropertySubType",
    "MlsStatus",
    "ContractStatusChangeDate",
    "PurchaseContractDate",
    "ListingContractDate",
]

homes = homes.drop(columns=columns_to_drop)

print(f"Remaining columns: {homes.shape[1]}")

Remaining columns: 38


In [140]:
# ============================================
# One-hot encode selected categorical variables
# ============================================

categorical_columns = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN",
    "Levels",
    "CountyOrParish",
    "StateOrProvince",
    "BuyerOfficeAOR",
    "BuyerAgentAOR",
    "ListAgentAOR",
]

homes = pd.get_dummies(
    homes,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print(f"Dataset shape after encoding: {homes.shape}")

Dataset shape after encoding: (399157, 319)


## Remove Data Leakage Features

Certain variables contain information that is too closely related to the target
variable (`ClosePrice`) and could artificially inflate model performance.

Following the internship project guidance, `ListPrice` and `OriginalListPrice` are removed
before model development to prevent data leakage. Identifier variables are also
removed because they uniquely identify listings rather than describe the
properties themselves.

In [141]:
# ============================================
# Remove data leakage and identifier variables
# ============================================

columns_to_drop = [
    "ListPrice",
    "OriginalListPrice",
    "ListingKey",
    "ListingKeyNumeric",
]

homes = homes.drop(columns=columns_to_drop)

print(f"Remaining columns: {homes.shape[1]}")

Remaining columns: 315


## Normalize Numerical Features

Many machine learning algorithms perform better when numerical variables are on
similar scales.

For the baseline model, numerical predictor variables will be standardized using
StandardScaler. For the baseline model, continuous numerical predictor variables are standardized
using `StandardScaler`. Binary indicator variables created through one-hot
encoding remain unchanged, and the target variable (`ClosePrice`) is excluded
from this process.

In [142]:
# ============================================
# Standardize continuous numerical features
# ============================================

from sklearn.preprocessing import StandardScaler

continuous_columns = [
    "Latitude",
    "Longitude",
    "LivingArea",
    "DaysOnMarket",
    "ParkingTotal",
    "LotSizeAcres",
    "YearBuilt",
    "StreetNumberNumeric",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "Stories",
    "LotSizeArea",
    "MainLevelBedrooms",
    "GarageSpaces",
    "AssociationFee",
    "LotSizeSquareFeet",
]

scaler = StandardScaler()

homes[continuous_columns] = scaler.fit_transform(
    homes[continuous_columns]
)

print(f"Scaled {len(continuous_columns)} continuous numerical features.")

Scaled 16 continuous numerical features.


## Create the Training and Testing Sets

To simulate a real-world prediction scenario, the dataset is divided
chronologically.

The most recent month of available data is reserved as the testing set. The X months immediately preceding the test month are used for training, where X is a configurable parameter that can be tuned during model development.

In [143]:
# ============================================
# Convert CloseDate to datetime
# ============================================

# The CRMLS files contain multiple date formats,
# so pandas infers the appropriate format for each value.
homes["CloseDate"] = pd.to_datetime(
    homes["CloseDate"],
    format="mixed"
)

print(f"Missing dates after conversion: {homes['CloseDate'].isna().sum()}")

# Identify the most recent month
latest_month = homes["CloseDate"].dt.to_period("M").max()

# Number of months to use for training.
# This value can be tuned during model development.
X = 5          # or 6, 12, etc.

train_start = latest_month - X

# Create training and testing sets
train_df = homes[
    (homes["CloseDate"].dt.to_period("M") >= train_start) &
    (homes["CloseDate"].dt.to_period("M") < latest_month)
].copy()

test_df = homes[
    homes["CloseDate"].dt.to_period("M") == latest_month
].copy()


print(f"Training rows: {len(train_df):,}")
print(f"Testing rows:  {len(test_df):,}")

Missing dates after conversion: 0
Training rows: 49,703
Testing rows:  12,024


## Export the Cleaned Datasets

The preprocessed training and testing datasets are saved as CSV files for use in
the baseline machine learning models developed in the next stage of the project.

In [144]:
# ============================================
# Export cleaned training and testing datasets
# ============================================

output_path = Path("../data")

train_df.to_csv(output_path / "train_cleaned.csv", index=False)
test_df.to_csv(output_path / "test_cleaned.csv", index=False)

print("Training dataset saved as train_cleaned.csv")
print("Testing dataset saved as test_cleaned.csv")

Training dataset saved as train_cleaned.csv
Testing dataset saved as test_cleaned.csv


In [145]:
print(f"Final dataset shape: {homes.shape}")
print(f"Training set: {train_df.shape}")
print(f"Testing set: {test_df.shape}")

Final dataset shape: (399157, 315)
Training set: (49703, 315)
Testing set: (12024, 315)


## Summary

In this notebook, the California CRMLS residential single-family dataset was
prepared for machine learning by:

- Removing empty columns and variables with excessive missing values.
- Imputing the remaining missing values.
- Removing identifier and data leakage variables.
- Encoding selected categorical variables.
- Standardizing continuous numerical features.
- Creating a time-based training and testing split using the most recent month as the test set and the configurable X preceding months as the training set.
- Exporting the cleaned datasets for model development.

These datasets are now ready for the baseline predictive models developed in the
next stage of the project.